# Magnus quantum time marching

Run both four-slice integrators and compare their counter-zero outcomes with the diagonal analytic solution.

## Build slices, then compose the integrator

The reusable `magnus` qfunc in [quantum.py](quantum.py) requests the static
phase-stream slice. Its checked helper supplies the controlled rotations and
inverse composition. The same module then uses Circuit to compose slices,
projector matrices and a counter permutation into the eleven-bit integrator.
The full solver therefore does not yet flow through one unified qfunc program.
Classical polynomial and phase fitting precede quantum execution.

Try inspecting a different slice interval with `expected_block(start, end)`
before changing the integrator schedule. The four-slice amplified and naive
experiments retain their distinct million-draw samples. Reading conditional
magnitudes from a histogram loses complex signs, so the independent coherent
state comparison remains necessary. A small sampled normalized error does not
by itself certify counter-zero probability, polynomial error or exact ODE error.


Set up the native FlagQuantum run and plots.

In [ ]:
from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.algorithms.quantum_differential_equations_solvers.time_marching.overview import (
    quantum,
    workflow,
)
from examples.synthesis.algorithms.quantum_differential_equations_solvers.time_marching.overview.main import (
    print_report,
    save_figures,
)


Keep the original duration, four time slices and two data bits.

In [ ]:
print("Duration and long slices:", quantum.END, quantum.SLICE_COUNT)
assert (quantum.END, quantum.SLICE_COUNT, quantum.WIDTH) == (2.0, 4, 11)


Solve the same diagonal differential equation analytically at 100 time points.

In [ ]:
result = workflow.run()
assert tuple(result.classical.shape) == (100, 4)
print("Unnormalized analytic final state:", result.classical[-1].tolist())


Evaluate the block using the original Y rotation angle 2(t + index).

In [ ]:
print("First time-dependent block:", quantum.expected_block(0, 0.5).tolist())


Average four left-endpoint samples per slice by the Hadamard-conjugated time register.

In [ ]:
print("Quadrature samples per slice:", 4)
print("Magnus polynomial block agreement:", result.magnus_block_error)


Compare both fresh exponential approximations to the original demonstration function.

In [ ]:
print("Exponential demonstration grid error:", result.scalar_fit_error)
figures = Path("notebook-figures")
save_figures(result, figures)
display(Image(filename=str(figures / "time-polynomials.png")))


Recompute the degree-six and degree-seven short-time phase sequences.

In [ ]:
print(
    "Short-slice phase counts:",
    result.exponential.even_fit.phases.numel(),
    result.exponential.odd_fit.phases.numel(),
)


Fit the degree-seven amplification response to 1.8 times its argument on the source interval.

In [ ]:
print("Amplifier interval error:", result.amplifier.interval_error)
assert result.amplifier.maximum_magnitude < 1


Verify the amplified integrator before reading its counter.

In [ ]:
assert result.amplified.state_error < 1e-10
print("Amplified success probability:", result.amplified.exact_prob)


Run one million finite draws for the amplified circuit and retain its counter-zero observations.

In [ ]:
print(
    "Amplified finite observations:", workflow.SHOTS, result.amplified.accepted_counts
)


Run the original unamplified comparator with a separate million draws.

In [ ]:
print("Naive finite observations:", workflow.SHOTS, result.naive.accepted_counts)


Postselect on counter zero and recover magnitudes from the measured joint frequencies.

In [ ]:
print("Amplified sampled magnitudes:", result.amplified.sampled_magnitudes.tolist())
print("Naive sampled magnitudes:", result.naive.sampled_magnitudes.tolist())


Compare the two finite reconstructions with the analytic diagonal solution.

In [ ]:
print_report(result)
display(Image(filename=str(figures / "time-marching.png")))


Require the amplified finite reconstruction to meet the source normalized error bound.

In [ ]:
assert result.amplified.normalized_error < 0.1
print("Amplified finite L2 error below 0.1:", result.amplified.normalized_error)
